# H-009 · GDELT sentiment

**Claim:** Higher / improving GDELT news tone (and attention) predicts higher forward open-to-open returns; primary horizon **5d**.

**Why it might work:** Media tone shapes perceived value and discretionary flow; attention scales intensity. Own-history z-scores strip permanently negative names.

**Data required:** GDELT daily `median_tone` / `n_articles` on the panel (merge on S1 `feature_date`). Hand-edit company names in `01_data/ingestion/alternative_data/sentiment/gdelt_company_name_map.csv` (console logs only `NOT_FOUND` / `AMBIGUOUS`).

**Store API:** `add_gdelt_sentiment_factors(..., feature_subset=..., sentiment_data_exists=...)` — IDs `tone`, `attention`, `abnormal_tone`, `abnormal_attention`, `tone_x_attention`, `tone_mom`. Default `sentiment_data_exists=False` fetches + merges inside the store; this notebook pre-merges for cache/diagnostics then passes `True`. No store `normalize`.

## Screen grid (research IS)

| ID | Params |
|---|---|
| `tone`, `attention`, `tone_x_attention` | `window=[1,5,10,21]` |
| `abnormal_tone`, `abnormal_attention` | `smooth=[1,5]` × `baseline=[60,126]` |
| `tone_mom` | `short=[1,5]` × `long=[10,21]` (all long > short) |

Expected **24** factor columns. Sort screen by `ic_5d` descending (positive tone → higher returns).

## Parquet caches

| Path | Role |
|------|------|
| `s1_factor_panel_train.parquet` | Research IS OHLCV (cold path) |
| `s1_h009_gdelt_daily.parquet` | Raw daily `date,ticker,median_tone,n_articles` |
| `s1_h009_gdelt_panel.parquet` | Enriched IS + all H-009 factor columns |

**Load gate:** if enriched panel exists and `FORCE_REBUILD` is False → skip fetch / merge / feature build.

**Fetch gate:** BigQuery only when daily parquet is missing (or `FORCE_REBUILD`). History pulls use **calendar-month** BQ windows (SQL match + daily median); resume caches live under `01_data/cache/alternative_data/gdelt/bq_scored/`.

**Smoke then full:** first validate with a short range (e.g. one month) via `fetch_gdelt_sentiment_daily(..., start_date=..., end_date=...)` before an overnight full `2015 → panel end` backfill. Watch for BigQuery free-tier quota errors (`free query bytes scanned`); resume continues from completed months.

**Invalidate** by deleting those parquets or setting `FORCE_REBUILD = True` after store/grid changes.

Evaluation: Alphalens open pivot (no `shift(-1)`); periods `(1,5,21)`. Full PDF tears only for hand-picked `TEAR_FACTORS` after §4.1.


## 0. Imports & Config


Resolve repo root; configure screen windows, cache paths, and Alphalens knobs. Set `FORCE_REBUILD = True` to ignore existing H-009 parquets and re-query BigQuery / rebuild factors. Prefer a one-month smoke fetch before the full history pull.


In [1]:
import os
import sys
import time

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import numpy as np
import pandas as pd
import alphalens as al

from data.ingestion.alternative_data.sentiment.gdelt_fetcher import (
    fetch_gdelt_sentiment_daily,
)
from data.processing.s1_feature_store import (
    GDELT_SENTIMENT_FEATURES,
    add_gdelt_sentiment_factors,
)

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

TRAIN_PANEL_PATH = os.path.join(
    ROOT, "01_data", "data_files", "s1_equities", "s1_factor_panel_train.parquet"
)
GDELT_DAILY_CACHE_PATH = os.path.join(
    ROOT, "01_data", "data_files", "s1_equities", "s1_h009_gdelt_daily.parquet"
)
GDELT_PANEL_CACHE_PATH = os.path.join(
    ROOT, "01_data", "data_files", "s1_equities", "s1_h009_gdelt_panel.parquet"
)
TEARSHEET_DIR = os.path.join(
    ROOT, "02_research", "notebooks", "s1_equities", "factor_tests", "tearsheets"
)

FORCE_REBUILD = False

FEATURE_SUBSET = list(GDELT_SENTIMENT_FEATURES)
WINDOWS = [1, 5, 10, 21]
SMOOTH_WINDOWS = [1, 5]
BASELINE_WINDOWS = [60, 126]
SHORT_WINDOWS = [1, 5]
LONG_WINDOWS = [10, 21]

# Expected columns when every ID above is multi-window:
# tone/attention/txa: 4 each; abnormal_*: 2x2 each; tone_mom: 4 → 24
EXPECTED_N_FACTORS = (
    3 * len(WINDOWS)
    + 2 * len(SMOOTH_WINDOWS) * len(BASELINE_WINDOWS)
    + len(SHORT_WINDOWS) * len(LONG_WINDOWS)
)

PERIODS = (1, 5, 21)
QUANTILES = 5
MAX_LOSS = 0.35

print(f"ROOT={ROOT}")
print(f"FORCE_REBUILD={FORCE_REBUILD}")
print(f"FEATURE_SUBSET={FEATURE_SUBSET}")
print(f"EXPECTED_N_FACTORS={EXPECTED_N_FACTORS}")
print(f"daily_cache={GDELT_DAILY_CACHE_PATH}")
print(f"panel_cache={GDELT_PANEL_CACHE_PATH}")


ROOT=c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio
FORCE_REBUILD=False
FEATURE_SUBSET=['tone', 'attention', 'abnormal_tone', 'abnormal_attention', 'tone_x_attention', 'tone_mom']
EXPECTED_N_FACTORS=24
daily_cache=c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\01_data\data_files\s1_equities\s1_h009_gdelt_daily.parquet
panel_cache=c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\01_data\data_files\s1_equities\s1_h009_gdelt_panel.parquet


## 1. Data Loading


If `s1_h009_gdelt_panel.parquet` exists → load and skip to evaluation. Otherwise load train IS, load/fetch daily GDELT (cache gate), rename `date`→`feature_date`, left-merge on `[feature_date, ticker]`.


In [2]:
use_panel_cache = (not FORCE_REBUILD) and os.path.isfile(GDELT_PANEL_CACHE_PATH)

if use_panel_cache:
    panel = pd.read_parquet(GDELT_PANEL_CACHE_PATH)
    panel["date"] = pd.to_datetime(panel["date"])
    if "feature_date" in panel.columns:
        panel["feature_date"] = pd.to_datetime(panel["feature_date"])
    panel["ticker"] = panel["ticker"].astype(str).str.strip().str.upper()
    FACTOR_COLS = sorted(c for c in panel.columns if c.startswith("gdelt_"))
    print(
        f"PANEL CACHE HIT: {GDELT_PANEL_CACHE_PATH}  "
        f"rows={len(panel):,}  factors={len(FACTOR_COLS)}"
    )
else:
    panel = pd.read_parquet(TRAIN_PANEL_PATH)
    required = {"date", "ticker", "open", "close", "feature_date"}
    missing = required - set(panel.columns)
    if missing:
        raise ValueError(f"train panel missing columns: {sorted(missing)}")
    if not panel["feature_date"].lt(panel["date"]).all():
        raise ValueError("feature_date must be strictly before date on every row")

    panel = panel.copy()
    panel["date"] = pd.to_datetime(panel["date"])
    panel["feature_date"] = pd.to_datetime(panel["feature_date"])
    panel["ticker"] = panel["ticker"].astype(str).str.strip().str.upper()
    tickers = sorted(panel["ticker"].unique().tolist())
    start = panel["feature_date"].min().date()
    end = panel["feature_date"].max().date()
    print(
        f"train panel: rows={len(panel):,}  tickers={len(tickers)}  "
        f"trade dates={panel['date'].nunique():,}  feature [{start} -> {end}]"
    )

    GDELT_MERGE = ["feature_date", "ticker", "median_tone", "n_articles"]
    use_daily = (not FORCE_REBUILD) and os.path.isfile(GDELT_DAILY_CACHE_PATH)
    if use_daily:
        gd = pd.read_parquet(GDELT_DAILY_CACHE_PATH)
        gd["date"] = pd.to_datetime(gd["date"])
        gd["ticker"] = gd["ticker"].astype(str).str.strip().str.upper()
        print(f"GDELT DAILY CACHE HIT: {GDELT_DAILY_CACHE_PATH} rows={len(gd):,}")
    else:
        print(f"GDELT DAILY CACHE MISS: BigQuery fetch [{start} -> {end}]")
        t0 = time.perf_counter()
        gd = fetch_gdelt_sentiment_daily(
            tickers,
            start_date=start,
            end_date=end,
            use_bigquery=True,
            live_n_files=0,
        )
        print(f"fetch wall={time.perf_counter() - t0:.1f}s  rows={len(gd):,}")
        gd["date"] = pd.to_datetime(gd["date"])
        gd["ticker"] = gd["ticker"].astype(str).str.strip().str.upper()
        os.makedirs(os.path.dirname(GDELT_DAILY_CACHE_PATH), exist_ok=True)
        gd[["date", "ticker", "median_tone", "n_articles"]].to_parquet(
            GDELT_DAILY_CACHE_PATH, index=False
        )
        print(f"Wrote {GDELT_DAILY_CACHE_PATH} rows={len(gd):,}")

    gd = gd.rename(columns={"date": "feature_date"})
    panel = panel.merge(gd[GDELT_MERGE], on=["feature_date", "ticker"], how="left")
    FACTOR_COLS = []  # filled in §3
    print(
        f"median_tone coverage={panel['median_tone'].notna().mean():.1%}  "
        f"n_articles coverage={panel['n_articles'].notna().mean():.1%}"
    )

panel.head()


train panel: rows=289,381  tickers=100  trade dates=2,915  feature [2010-01-04 -> 2021-08-02]
GDELT DAILY CACHE MISS: BigQuery fetch [2010-01-04 -> 2021-08-02]


GDELT BQ:   0%|          | 0/79 [00:00<?, ?month/s]

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery S

GDELT_MAP	NOT_FOUND	AET
GDELT_MAP	AMBIGUOUS	AMERICAN
GDELT_MAP	NOT_FOUND	CLEVELAND-CLIFFS INC.
GDELT_MAP	NOT_FOUND	CAPITAL ONE FINANCIAL CORP
GDELT_MAP	NOT_FOUND	ESRX
GDELT_MAP	NOT_FOUND	ELI LILLY & Co
GDELT_MAP	NOT_FOUND	NOV Inc.
GDELT_MAP	NOT_FOUND	TJX COMPANIES INC /DE/
GDELT_MAP	NOT_FOUND	TWX
fetch wall=530.8s  rows=203,087
Wrote c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\01_data\data_files\s1_equities\s1_h009_gdelt_daily.parquet rows=203,087
median_tone coverage=48.8%  n_articles coverage=48.8%


,date,ticker,open,high,low,close,volume,feature_date,fwd_ret_1,fwd_ret_5,fwd_ret_21,median_tone,n_articles
0,2010-01-05,AAPL,6.424143,6.421146,6.357683,6.406478,493729600.0,2010-01-04,-0.001025,-0.025210,-0.083271,NaN,NaN
1,2010-01-06,AAPL,6.417558,6.453779,6.383729,6.417557,601904800.0,2010-01-05,-0.012268,-0.030367,-0.101456,NaN,NaN
2,2010-01-07,AAPL,6.338825,6.443003,6.308892,6.315478,552160000.0,2010-01-06,-0.006848,-0.007745,-0.075844,NaN,NaN
3,2010-01-08,AAPL,6.295419,6.346309,6.258000,6.303801,477131200.0,2010-01-07,0.011888,0.002996,-0.066001,NaN,NaN
4,2010-01-11,AAPL,6.370258,6.346310,6.258300,6.345711,447610800.0,2010-01-08,-0.016965,-0.021005,-0.079464,NaN,NaN


## 2. Data Cleaning & Engineering


Coverage diagnostics only — no winsorize in the store. Paste `GDELT_MAP\tNOT_FOUND|AMBIGUOUS\t...` company names into `gdelt_company_name_map.csv` and re-fetch (`FORCE_REBUILD=True`) if mapping is thin.


In [3]:
if use_panel_cache:
    print("PANEL CACHE HIT — skip §2 diagnostics (raw coverage already baked into factors)")
else:
    tone_cov = panel.groupby("ticker")["median_tone"].apply(lambda s: s.notna().mean())
    art_cov = panel.groupby("ticker")["n_articles"].apply(lambda s: s.notna().mean())
    print("median_tone coverage by ticker (describe):")
    print(tone_cov.describe())
    print("\nn_articles coverage by ticker (describe):")
    print(art_cov.describe())

    thin = tone_cov[tone_cov < 0.05].sort_values()
    print(f"\ntickers with median_tone coverage < 5%: {len(thin)}")
    if len(thin):
        print(thin.head(20).to_string())

    print("\nmedian_tone describe (non-null):")
    print(panel["median_tone"].dropna().describe())
    print("\nn_articles describe (non-null):")
    print(panel["n_articles"].dropna().describe())


median_tone coverage by ticker (describe):
count    100.000000
mean       0.484700
std        0.175282
min        0.000000
25%        0.555317
50%        0.557461
75%        0.557461
max        0.557461
Name: median_tone, dtype: float64

n_articles coverage by ticker (describe):
count    100.000000
mean       0.484700
std        0.175282
min        0.000000
25%        0.555317
50%        0.557461
75%        0.557461
max        0.557461
Name: n_articles, dtype: float64

tickers with median_tone coverage < 5%: 9
ticker
AET     0.00000
CLF     0.00000
COF     0.00000
ESRX    0.00000
NOV     0.00000
LLY     0.00000
TJX     0.00000
TWX     0.00000
APA     0.01578

median_tone describe (non-null):
count    141290.000000
mean          0.075072
std           1.385959
min         -15.593220
25%          -0.634249
50%           0.169779
75%           0.866218
max          16.307692
Name: median_tone, dtype: float64

n_articles describe (non-null):
count    141290.000000
mean       1205.734631
st

## 3. Modeling / Signal Construction


Cold path: `add_gdelt_sentiment_factors(..., sentiment_data_exists=True)` with the full screen grid (daily already merged above), then write `s1_h009_gdelt_panel.parquet` before any Alphalens work.


In [4]:
if use_panel_cache:
    print(f"PANEL CACHE HIT — reusing {len(FACTOR_COLS)} factor columns")
else:
    t0 = time.perf_counter()
    panel = add_gdelt_sentiment_factors(
        panel,
        feature_subset=FEATURE_SUBSET,
        sentiment_data_exists=True,
        window=WINDOWS,
        smooth_window=SMOOTH_WINDOWS,
        baseline_window=BASELINE_WINDOWS,
        short_window=SHORT_WINDOWS,
        long_window=LONG_WINDOWS,
    )
    FACTOR_COLS = sorted(c for c in panel.columns if c.startswith("gdelt_"))
    print(
        f"features wall={time.perf_counter() - t0:.1f}s  "
        f"n_factors={len(FACTOR_COLS)} (expected {EXPECTED_N_FACTORS})"
    )
    if len(FACTOR_COLS) != EXPECTED_N_FACTORS:
        raise ValueError(
            f"unexpected factor count {len(FACTOR_COLS)} != {EXPECTED_N_FACTORS}: "
            f"{FACTOR_COLS}"
        )

    os.makedirs(os.path.dirname(GDELT_PANEL_CACHE_PATH), exist_ok=True)
    panel.to_parquet(GDELT_PANEL_CACHE_PATH, index=False)
    print(f"Wrote {GDELT_PANEL_CACHE_PATH} rows={len(panel):,}")

print("FACTOR_COLS:")
for c in FACTOR_COLS:
    print(f"  {c}")


features wall=9.3s  n_factors=24 (expected 24)
Wrote c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\01_data\data_files\s1_equities\s1_h009_gdelt_panel.parquet rows=289,381
FACTOR_COLS:
  gdelt_abnormal_attention_1_126
  gdelt_abnormal_attention_1_60
  gdelt_abnormal_attention_5_126
  gdelt_abnormal_attention_5_60
  gdelt_abnormal_tone_1_126
  gdelt_abnormal_tone_1_60
  gdelt_abnormal_tone_5_126
  gdelt_abnormal_tone_5_60
  gdelt_attention_1
  gdelt_attention_10
  gdelt_attention_21
  gdelt_attention_5
  gdelt_tone_1
  gdelt_tone_10
  gdelt_tone_21
  gdelt_tone_5
  gdelt_tone_mom_1_10
  gdelt_tone_mom_1_21
  gdelt_tone_mom_5_10
  gdelt_tone_mom_5_21
  gdelt_tone_x_attention_1
  gdelt_tone_x_attention_10
  gdelt_tone_x_attention_21
  gdelt_tone_x_attention_5


## 4. Evaluation


### 4.1 IC / spread screen

Alphalens helpers + screen every `gdelt_*` column at periods `(1,5,21)`. Primary sort key: **`ic_5d` descending**. Also print best column per feature family.


In [5]:
def parse_gdelt_factor_name(col: str) -> dict | None:
    """Map `gdelt_{family}_{windows...}` → {feature, windows}."""
    if not isinstance(col, str) or not col.startswith("gdelt_"):
        return None
    rest = col[len("gdelt_") :]
    families = (
        "abnormal_attention",
        "abnormal_tone",
        "tone_x_attention",
        "tone_mom",
        "attention",
        "tone",
    )
    for fam in families:
        if rest == fam or rest.startswith(fam + "_"):
            suffix = rest[len(fam) :].lstrip("_")
            parts = []
            if suffix:
                for tok in suffix.split("_"):
                    if tok.isdigit():
                        parts.append(int(tok))
                    else:
                        return None
            return {"feature": fam, "windows": parts}
    return None


def to_alphalens_prices(panel: pd.DataFrame) -> pd.DataFrame:
    """Wide open matrix for Alphalens (trade-date panel; entry at open)."""
    prices = panel.pivot(index="date", columns="ticker", values="open")
    prices.index = pd.to_datetime(prices.index)
    return prices.sort_index()


def to_alphalens_factor(panel: pd.DataFrame, col: str) -> pd.Series:
    """MultiIndex (date, ticker) factor series for Alphalens."""
    factor = panel.set_index(["date", "ticker"])[col].dropna()
    factor.index = factor.index.set_levels(
        pd.to_datetime(factor.index.levels[0]), level=0
    )
    return factor.sort_index()


def _period_label(period_index: pd.Index, period: int, position: int):
    """Match Alphalens period label ('1D', '5D', ...) or fall back by position."""
    for c in (f"{period}D", f"{period}d", period, str(period)):
        if c in period_index:
            return c
    return period_index[position]


def factor_screen_metrics(
    factor: pd.Series,
    prices: pd.DataFrame,
    *,
    periods: tuple[int, ...] = PERIODS,
    quantiles: int = QUANTILES,
    max_loss: float = MAX_LOSS,
) -> dict:
    """Mean IC and Q5-Q1 mean return spread for each forward period."""
    factor_data = al.utils.get_clean_factor_and_forward_returns(
        factor=factor,
        prices=prices,
        quantiles=quantiles,
        periods=periods,
        max_loss=max_loss,
    )
    mean_ic = al.performance.mean_information_coefficient(factor_data)
    mean_ret, _ = al.performance.mean_return_by_quantile(factor_data, demeaned=True)

    row = {}
    for i, p in enumerate(periods):
        ic_key = _period_label(mean_ic.index, p, i)
        ret_key = _period_label(mean_ret.columns, p, i)
        row[f"ic_{p}d"] = float(mean_ic.loc[ic_key])
        q_hi, q_lo = mean_ret.index.max(), mean_ret.index.min()
        row[f"spread_{p}d"] = float(
            mean_ret.loc[q_hi, ret_key] - mean_ret.loc[q_lo, ret_key]
        )
    return row


def run_full_tear(
    panel: pd.DataFrame,
    factor_col: str,
    prices: pd.DataFrame,
    *,
    periods: tuple[int, ...] = PERIODS,
    quantiles: int = QUANTILES,
    max_loss: float = MAX_LOSS,
    tearsheet_dir: str = TEARSHEET_DIR,
):
    """Build factor_data, run Alphalens full tear, save figs to multi-page PDF.

    Alphalens calls plt.show() after each plot, which clears figures under Agg.
    Temporarily replace plt.show so each figure is written into the PDF before close.
    """
    if factor_col not in panel.columns:
        raise ValueError(
            f"{factor_col!r} not in panel - pick a screened column "
            f"(available: {FACTOR_COLS})"
        )
    plt.close("all")
    factor_data = al.utils.get_clean_factor_and_forward_returns(
        factor=to_alphalens_factor(panel, factor_col),
        prices=prices,
        quantiles=quantiles,
        periods=periods,
        max_loss=max_loss,
    )

    os.makedirs(tearsheet_dir, exist_ok=True)
    out_path = os.path.join(tearsheet_dir, f"H-009_{factor_col}.pdf")
    pdf = PdfPages(out_path)
    n_pages = 0
    _original_show = plt.show

    def _show_and_savefig(*args, **kwargs):
        nonlocal n_pages
        for num in list(plt.get_fignums()):
            fig = plt.figure(num)
            if fig.axes:
                pdf.savefig(fig, bbox_inches="tight")
                n_pages += 1
        plt.close("all")

    plt.show = _show_and_savefig
    try:
        al.tears.create_full_tear_sheet(factor_data, long_short=True)
        _show_and_savefig()
    finally:
        plt.show = _original_show
        pdf.close()
        plt.close("all")

    print(f"Wrote {out_path} ({n_pages} pages)")
    return factor_data


In [6]:
t0 = time.perf_counter()
prices = to_alphalens_prices(panel)
rows = []
for col in FACTOR_COLS:
    meta = parse_gdelt_factor_name(col) or {}
    metrics = factor_screen_metrics(to_alphalens_factor(panel, col), prices)
    rows.append(
        {
            "factor": col,
            "feature_family": meta.get("feature"),
            "windows": meta.get("windows"),
            **metrics,
        }
    )

summary = (
    pd.DataFrame(rows)
    .sort_values("ic_5d", ascending=False)
    .reset_index(drop=True)
)
print(f"Alphalens screen wall={time.perf_counter() - t0:.1f}s  n={len(summary)}")

best_by_family = (
    summary.sort_values("ic_5d", ascending=False)
    .groupby("feature_family", as_index=False)
    .first()
    .sort_values("ic_5d", ascending=False)
    .reset_index(drop=True)
)

print("\n=== Best by feature family (ic_5d) ===")
print(best_by_family.to_string())

print("\n=== Full screen (sorted by ic_5d) ===")
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(summary)


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any n

Alphalens screen wall=176.1s  n=24

=== Best by feature family (ic_5d) ===
       feature_family                         factor  windows     ic_1d  spread_1d     ic_5d  spread_5d    ic_21d  spread_21d
0    tone_x_attention      gdelt_tone_x_attention_21     [21]  0.016603   0.000348  0.026606   0.001534  0.046263    0.007312
1                tone                  gdelt_tone_21     [21]  0.016343   0.000235  0.024964   0.001331  0.041887    0.006199
2           attention              gdelt_attention_5      [5]  0.005463   0.000162  0.012285   0.000903  0.018037    0.002527
3  abnormal_attention  gdelt_abnormal_attention_5_60  [5, 60]  0.001504  -0.000050  0.008809   0.000514 -0.003262   -0.000358
4            tone_mom            gdelt_tone_mom_1_10  [1, 10]  0.007447   0.000430  0.005308   0.000680  0.001705    0.000524
5       abnormal_tone       gdelt_abnormal_tone_1_60  [1, 60]  0.006208   0.000376  0.002654   0.000471 -0.003633   -0.000609

=== Full screen (sorted by ic_5d) ===


,factor,feature_family,windows,ic_1d,spread_1d,ic_5d,spread_5d,ic_21d,spread_21d
0,gdelt_tone_x_attention_21,tone_x_attention,[21],0.016603,0.000348,0.026606,0.001534,0.046263,0.007312
1,gdelt_tone_21,tone,[21],0.016343,0.000235,0.024964,0.001331,0.041887,0.006199
2,gdelt_tone_x_attention_10,tone_x_attention,[10],0.014394,0.000228,0.022894,0.001222,0.042395,0.007727
3,gdelt_tone_10,tone,[10],0.013914,0.000237,0.020993,0.001197,0.038041,0.005909
4,gdelt_tone_x_attention_5,tone_x_attention,[5],0.014833,0.000297,0.019784,0.001291,0.035954,0.006150
5,gdelt_tone_5,tone,[5],0.014582,0.000301,0.017479,0.000881,0.031505,0.004795
6,gdelt_tone_x_attention_1,tone_x_attention,[1],0.011689,0.000352,0.017413,0.001529,0.025576,0.004622
7,gdelt_tone_1,tone,[1],0.012017,0.000368,0.015154,0.001216,0.021241,0.003164
8,gdelt_attention_5,attention,[5],0.005463,0.000162,0.012285,0.000903,0.018037,0.002527
9,gdelt_attention_1,attention,[1],0.005420,0.000055,0.011745,0.000424,0.019223,0.002285


### 4.2 Full tear sheet (manual pick)

After reviewing §4.1, fill `TEAR_FACTORS` with column names to dump. Empty list = skip PDFs.


In [7]:
# Edit after reviewing §4.1 — leave empty until you pick winners
TEAR_FACTORS = [
    "gdelt_tone_x_attention_21",
    "gdelt_tone_21",
    "gdelt_attention_5",
    "gdelt_abnormal_attention_5_60",
]

for tear_col in TEAR_FACTORS:
    print(f"\n===== Tear sheet: {tear_col} =====")
    run_full_tear(panel, tear_col, prices)

if not TEAR_FACTORS:
    print("TEAR_FACTORS empty — skipping full tears (fill after §4.1)")



===== Tear sheet: gdelt_tone_x_attention_21 =====


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any n

Quantiles Statistics


,min,max,mean,std,count,count %
factor_quantile,,,,,,
1.0,-32.918770,0.000000,-5.967682,3.856522,26303,20.559177
2.0,-9.506773,2.018512,-0.866699,1.187458,29082,22.731323
3.0,-4.992522,3.280227,0.853237,1.006806,21414,16.737795
4.0,-1.422666,5.606131,2.632252,0.889134,25205,19.700949
5.0,0.678011,27.919982,5.636633,2.192864,25934,20.270756


Returns Analysis


...

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\performance.py:118: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  grouper.append(pd.Grouper(freq=by_time))
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:928: UserWarning: Skipping return periods that aren't exact multiples of days.
  warnings.warn(


Turnover Analysis


,1D,5D,21D
Quantile 1 Mean Turnover,0.246,0.663,0.783
Quantile 2 Mean Turnover,0.494,0.768,0.798
Quantile 3 Mean Turnover,0.532,0.786,0.806
Quantile 4 Mean Turnover,0.480,0.766,0.804
Quantile 5 Mean Turnover,0.230,0.682,0.807


,1D,5D,21D
Mean Factor Rank Autocorrelation,0.889,0.267,0.018


Wrote c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\02_research\notebooks\factor_tests\tearsheets\H-009_gdelt_abnormal_attention_5_60.pdf (3 pages)


## 5. Wrap-up / next steps

- Record variants tried vs best in `hypothesis_log.md` (pre-registration).
- Keep/kill rests on correlation to existing kept factors and nested GBM gain, not standalone IC alone.
- Re-run with `FORCE_REBUILD=True` after editing `gdelt_company_name_map.csv` if coverage was thin.
- Fill `TEAR_FACTORS` and re-run §4.2 for PDF dumps under `tearsheets/H-009_{col}.pdf`.
